In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paper formal baseline worker

Frozen formal entry for `t2smark`. Opening this notebook does not itself authorize execution; run only after explicit formal-experiment authorization.


In [ ]:
import json, os, pathlib, subprocess, sys
from google.colab import userdata

REPO = 'https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_EXACT = 'e4cf4ed2738cb91204695efbf9fb6ce35858b5f7'
METHOD = 't2smark'
JOB_ID = 'paper-baseline-t2smark-v1'
checkout = pathlib.Path('/content/cegwm-baseline-formal')
runtime_root = pathlib.Path('/content/cegwm-baseline-runtime') / METHOD
drive_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/PaperFormal-V1')

if not checkout.exists(): subprocess.run(['git', 'clone', REPO, str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', EXPECTED_EXACT], check=True)
head = subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', '-C', str(checkout), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
assert head == EXPECTED_EXACT and not dirty
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.32.0', 'transformers==4.45.2', 'accelerate==1.1.1', 'huggingface_hub==0.26.2', 'safetensors==0.4.5', 'sentencepiece==0.2.0', 'lpips', 'torchmetrics'], check=True)
child_env = dict(os.environ)
child_env['PYTHONPATH'] = str(checkout / 'src') + os.pathsep + str(checkout)
child_env['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
assert child_env['HF_TOKEN']
command = [sys.executable, '-m', 'experiments.run_paper_baseline_worker', '--method', METHOD, '--job-id', JOB_ID, '--expected-exact', EXPECTED_EXACT, '--drive-root', str(drive_root / 'baselines'), '--runtime-root', str(runtime_root)]
completed = subprocess.run(command, cwd=checkout, env=child_env, check=False)
if completed.returncode != 0:
    state_path = drive_root / 'baselines' / JOB_ID / 'job_state.json'
    state = json.loads(state_path.read_text(encoding='utf-8')) if state_path.exists() else {}
    raise RuntimeError(f"worker exited {completed.returncode}: error_code={state.get('error_code')} error={state.get('error')}")
final_path = drive_root / 'baselines' / JOB_ID / 'method_final.json'
public = json.loads(final_path.read_text(encoding='utf-8'))
print({'method_id': public['method_id'], 'status': public['status'], 'result_package_produced': public['result_package_produced']})
